In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "../data",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "../data",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

data_aug = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomBrightness(0.5),
        tf.keras.layers.RandomContrast(0.5),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
    ]
)

normalization = tf.keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(lambda x, y: (normalization(data_aug(x)), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))

AUTO_TUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTO_TUNE)
val_ds = val_ds.prefetch(buffer_size=AUTO_TUNE)

Found 3846 files belonging to 2 classes.
Using 3077 files for training.
Found 3846 files belonging to 2 classes.
Using 769 files for validation.


In [ ]:
base_model = MobileNetV2(
    weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
x = base_model.output
x = layers.GlobalMaxPooling2D()(x)
x = layers.Dense(129, activation="relu")(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs=base_model.input, outputs=output)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 172s 2s/step - accuracy: 0.8599 - loss: 0.4801 - val_accuracy: 0.9571 - val_loss: 0.1197
Epoch 2/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 151s 2s/step - accuracy: 0.9191 - loss: 0.2333 - val_accuracy: 0.9662 - val_loss: 0.0944
Epoch 3/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 165s 2s/step - accuracy: 0.9327 - loss: 0.1799 - val_accuracy: 0.9831 - val_loss: 0.0596
Epoch 4/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - accuracy: 0.9415 - loss: 0.1543 - val_accuracy: 0.9766 - val_loss: 0.0840
Epoch 5/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 128s 1s/step - accuracy: 0.9435 - loss: 0.1492 - val_accuracy: 0.9857 - val_loss: 0.0452
Epoch 6/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.9402 - loss: 0.1629 - val_accuracy: 0.9389 - val_loss: 0.1404
Epoch 7/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - accuracy: 0.9412 - loss: 0.1590 - val_accuracy: 0.9805 - val_loss: 0.0623
Epoch 8/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.9552 - loss: 0.1240 - val_accuracy: 0.9896 - v

In [ ]:
# model saving
model.save('../models/base_model.keras')